# Medical Insurance Cost Prediction — ML Pipeline & Walkthrough

**Objective:** Build, evaluate, and serialize an end-to-end Machine Learning regression model that estimates individual medical insurance charges based on demographic and biometric health attributes.

### Pipeline Stages:
1. **Data Ingestion & Integrity Check**
2. **Exploratory Data Analysis (EDA)**
3. **Feature Preprocessing Pipeline (Scikit-Learn ColumnTransformer)**
4. **Train / Test Split (Zero Data Leakage)**
5. **Model Benchmarking: Linear Regression, Random Forest, Gradient Boosting**
6. **Evaluation Metrics (MAE, MSE, RMSE, R²)**
7. **Diagnostic Residual & Error Analysis**
8. **Feature Importance Interpretation**
9. **Pipeline Serialization with Joblib**

In [ ]:
# Imports
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries successfully imported!")

## 1. Load Dataset & Inspection

In [ ]:
# Load insurance.csv (fetch if not present)
csv_path = "../insurance.csv" if os.path.exists("../insurance.csv") else "insurance.csv"
if not os.path.exists(csv_path):
    url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
    urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Data Types & Null Inspection
print("=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Duplicate Records ===")
duplicates = df.duplicated().sum()
print(f"Duplicates found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates dropped.")

df.info()

In [ ]:
# Statistical Summary
df.describe().T

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Visualizing Key Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['charges'], bins=30, color='#0D9488', edgecolor='black')
axes[0].set_title("Distribution of Charges ($)")
axes[0].set_xlabel("Charges")

axes[1].hist(df['bmi'], bins=30, color='#06B6D4', edgecolor='black')
axes[1].set_title("Distribution of BMI")
axes[1].set_xlabel("BMI")

axes[2].hist(df['age'], bins=25, color='#3B82F6', edgecolor='black')
axes[2].set_title("Distribution of Age")
axes[2].set_xlabel("Age")

plt.tight_layout()
plt.show()

In [ ]:
# Smoker vs Non-Smoker Charges
smoker_group = df.groupby('smoker')['charges'].agg(['mean', 'median', 'count'])
print("Charges by Smoking Status:")
print(smoker_group)

## 3. Data Preprocessing Pipeline
Build a robust `ColumnTransformer` with `StandardScaler` for numeric columns (`age`, `bmi`, `children`) and `OneHotEncoder` for categorical columns (`sex`, `smoker`, `region`).

In [ ]:
numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X = df[numeric_features + categorical_features]
y = df['charges']

# 80-20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

## 4. Model Training & Comparison

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42)
}

results = []
trained_pipelines = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', estimator)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "MAE ($)": round(mae, 2),
        "MSE": round(mse, 2),
        "RMSE ($)": round(rmse, 2),
        "R² Score": round(r2, 4)
    })
    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results)
results_df

## 5. Champion Model Analysis & Feature Importance

In [ ]:
champion_pipeline = trained_pipelines["Gradient Boosting"]
fitted_gb = champion_pipeline.named_steps['regressor']
fitted_prep = champion_pipeline.named_steps['preprocessor']

cat_encoder = fitted_prep.named_transformers_['cat']
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + cat_feature_names

feat_importances = pd.Series(fitted_gb.feature_importances_, index=all_feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind='barh', color='#0D9488')
plt.title("Gradient Boosting Feature Importance")
plt.xlabel("Relative Importance Score")
plt.tight_layout()
plt.show()

## 6. Model Serialization

In [ ]:
os.makedirs("../models", exist_ok=True)
save_target = "../models/insurance_model.pkl"
joblib.dump(champion_pipeline, save_target)
print(f"Successfully saved champion pipeline to: {save_target}")

# Verification test load
loaded_model = joblib.load(save_target)
sample_input = pd.DataFrame([{
    "age": 35,
    "sex": "male",
    "bmi": 27.4,
    "children": 2,
    "smoker": "no",
    "region": "southwest"
}])
pred = loaded_model.predict(sample_input)[0]
print(f"Test Prediction for 35yo Non-Smoker: ${pred:,.2f}")